# Step 1 — Merge Tables & Engineer Features

This notebook joins the 6 CSVs into **one training-ready table**: `07_model_training_table.csv`.
Each row = one maintenance block, enriched with everything relevant looked up from the other files.

**Inputs required (same folder as this notebook):**
- `03_section_traffic_derived.csv`
- `04_asset_register.csv`
- `05_maintenance_blocks.csv`
- `06_daily_asset_availability.csv`

(`01_stations_master.csv` and `02_train_timetable.csv` already fed into building file 3, so they aren't needed again here.)


In [8]:
import pandas as pd
import numpy as np

blocks = pd.read_csv(
    "data/synthetic/05_maintenance_blocks.csv",
    parse_dates=["date"]
)

traffic = pd.read_csv(
    "data/derived/03_section_traffic_derived.csv"
)

assets = pd.read_csv(
    "data/synthetic/04_asset_register.csv"
)

avail = pd.read_csv(
    "data/synthetic/06_daily_asset_availability.csv",
    parse_dates=["date"]
)

FileNotFoundError: [Errno 2] No such file or directory: 'data/synthetic/05_maintenance_blocks.csv'

## 1. Attach real traffic level at the block's section + start hour

Every block has a `section_id` and a `planned_start` time. We extract the hour and join to `03_section_traffic_derived.csv` on `(section_id, hour_of_day)`.

If there's no matching row, that genuinely means **no real train was ever scheduled through that section at that hour** — so we fill it as 0 trains / LOW traffic rather than treating it as missing data.


In [2]:
blocks["start_hour"] = blocks["planned_start"].str.split(":").str[0].astype(int)

df = blocks.merge(
    traffic[["section_id", "hour_of_day", "trains_count", "traffic_level"]],
    left_on=["section_id", "start_hour"], right_on=["section_id", "hour_of_day"],
    how="left"
).drop(columns=["hour_of_day"])

df["trains_count"] = df["trains_count"].fillna(0)
df["traffic_level"] = df["traffic_level"].fillna("LOW")
df[["section_id", "start_hour", "trains_count", "traffic_level"]].head()


,section_id,start_hour,trains_count,traffic_level
0,KOG-RIS,0,1.0,LOW
1,MS-MSB,0,0.0,LOW
2,UBC-UMB,2,2.0,LOW
3,ST-UDN,0,0.0,LOW
4,MRGA-NLDA,22,0.0,LOW


## 2. Attach asset-type availability for that date

The maintenance table doesn't say *which specific asset* was used — only the `required_asset_type`. So instead of joining to one asset, we compute: **"what % of assets of this type were available on this date?"**, using the real sampled records in `06_daily_asset_availability.csv`.

Not every asset has a sampled record for every date (the availability file only samples 40 days per asset). Where there's no record for that exact date, we fall back to that asset type's overall average availability — and we keep a flag column (`had_availability_record`) so the model can tell real observations apart from fallback estimates.


In [ ]:
avail_with_type = avail.merge(assets[["asset_id", "asset_type"]], on="asset_id", how="left")

daily_type_avail = (avail_with_type.groupby(["date", "asset_type"])["available"]
                     .apply(lambda s: (s == "YES").mean())
                     .reset_index(name="asset_type_availability_pct"))

overall_type_avail = (avail_with_type.groupby("asset_type")["available"]
                       .apply(lambda s: (s == "YES").mean())
                       .reset_index(name="overall_avg_availability_pct"))

df = df.merge(daily_type_avail, left_on=["date", "required_asset_type"],
              right_on=["date", "asset_type"], how="left").drop(columns=["asset_type"])
df = df.merge(overall_type_avail, left_on="required_asset_type", right_on="asset_type", how="left").drop(columns=["asset_type"])

df["had_availability_record"] = df["asset_type_availability_pct"].notna().astype(int)
df["asset_type_availability_pct"] = df["asset_type_availability_pct"].fillna(df["overall_avg_availability_pct"])
df = df.drop(columns=["overall_avg_availability_pct"])

print(f"{df['had_availability_record'].mean()*100:.1f}% rows had a real sampled record for that exact date")
df[["required_asset_type", "asset_type_availability_pct", "had_availability_record"]].head()


70.5% rows had a real sampled record for that exact date


,required_asset_type,asset_type_availability_pct,had_availability_record
0,Tower Wagon,1.0,1
1,Rail Crane,1.0,1
2,Rail Crane,1.0,1
3,Ballast Regulator,0.0,1
4,Dynamic Track Stabilizer,1.0,1


## 3. How many of the needed asset are homed near this section?

A simple count: for this block's section and required asset type, how many assets are registered as living there (`home_section_id` in `04_asset_register.csv`)? Zero doesn't mean no data — it usually means the nearest asset has to travel from elsewhere, which is itself a useful signal.


In [4]:
asset_counts = (assets.groupby(["home_section_id", "asset_type"]).size()
                .reset_index(name="assets_of_type_in_section"))

df = df.merge(asset_counts, left_on=["section_id", "required_asset_type"],
              right_on=["home_section_id", "asset_type"], how="left").drop(columns=["home_section_id", "asset_type"])
df["assets_of_type_in_section"] = df["assets_of_type_in_section"].fillna(0)
df[["section_id", "required_asset_type", "assets_of_type_in_section"]].head()


,section_id,required_asset_type,assets_of_type_in_section
0,KOG-RIS,Tower Wagon,0.0
1,MS-MSB,Rail Crane,0.0
2,UBC-UMB,Rail Crane,1.0
3,ST-UDN,Ballast Regulator,1.0
4,MRGA-NLDA,Dynamic Track Stabilizer,0.0


## 4. Historical overrun rate per section — WITHOUT leakage

This is the most important feature, and the easiest one to get wrong. We want: *"how often has this section overrun in the past?"* — but only counting blocks that happened **before** the current one. If we used the whole section's average overrun rate including future blocks, the model would be cheating (seeing the answer baked into a feature).

We sort by date, then use `shift(1).expanding().mean()` per section — this only looks backward.


In [5]:
df = df.sort_values(["section_id", "date"]).reset_index(drop=True)

df["section_historical_overrun_rate"] = (
    df.groupby("section_id")["overrun_flag"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

# A section's very first block has no history yet -> fall back to the global running average
global_rate_so_far = df["overrun_flag"].expanding().mean().shift(1)
df["section_historical_overrun_rate"] = df["section_historical_overrun_rate"].fillna(global_rate_so_far)
df["section_historical_overrun_rate"] = df["section_historical_overrun_rate"].fillna(df["overrun_flag"].mean())

df[["section_id", "date", "overrun_flag", "section_historical_overrun_rate"]].head(10)


,section_id,date,overrun_flag,section_historical_overrun_rate
0,ABB-NALR,2023-02-24,1,0.553333
1,ABB-NALR,2023-04-07,0,1.000000
2,ABB-NALR,2023-05-09,0,0.500000
3,ABB-NALR,2023-08-22,1,0.333333
4,ABB-NALR,2023-12-29,1,0.500000
5,ABB-NALR,2024-06-29,1,0.600000
6,ABB-NALR,2024-10-31,1,0.666667
7,ABB-NALR,2024-11-30,0,0.714286
8,ABB-NALR,2025-01-06,1,0.625000
9,ABB-NALR,2025-01-18,1,0.666667


## 5. Calendar features

Simple but useful: day of week, month, weekend flag.


In [6]:
df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)
df[["date", "day_of_week", "month", "is_weekend"]].head()


,date,day_of_week,month,is_weekend
0,2023-02-24,Friday,2,0
1,2023-04-07,Friday,4,0
2,2023-05-09,Tuesday,5,0
3,2023-08-22,Tuesday,8,0
4,2023-12-29,Friday,12,0


## 6. Assemble the final table

⚠️ **Important for Step 2 (modeling):** `actual_duration_min` is kept here for reference, but it's **leakage** — it's literally `planned_duration_min + overrun_min`, so it must be dropped before training a model to predict `overrun_min`. It's included in this table only so you can double-check the math, not to feed into a model.


In [7]:
df = df.sort_values(["date", "section_id"]).reset_index(drop=True)

cols = ["block_id", "section_id", "date", "day_of_week", "month", "is_weekend",
        "maintenance_type", "required_asset_type", "priority", "weather",
        "start_hour", "trains_count", "traffic_level",
        "assets_of_type_in_section", "asset_type_availability_pct", "had_availability_record",
        "section_historical_overrun_rate",
        "planned_duration_min",
        "overrun_min", "overrun_flag",            # <- TARGETS for Step 2
        "actual_duration_min"]                     # <- LEAKAGE, reference only, DROP before training

df = df[cols]
df.to_csv(f"{OUT}/07_model_training_table.csv", index=False)

print("Final shape:", df.shape)
print("\nNull check (should be empty):")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()


Final shape: (15000, 21)

Null check (should be empty):
Series([], dtype: int64)


,block_id,section_id,date,day_of_week,month,is_weekend,maintenance_type,required_asset_type,priority,weather,...,trains_count,traffic_level,assets_of_type_in_section,asset_type_availability_pct,had_availability_record,section_historical_overrun_rate,planned_duration_min,overrun_min,overrun_flag,actual_duration_min
0,B03841,BAM-PSA,2023-01-01,Sunday,1,1,Ballast Renewal,Ballast Regulator,Low,Light Rain,...,2.0,LOW,0.0,0.0,1,0.557272,221.8,0.2,0,222.0
1,B12744,BE-SPN,2023-01-01,Sunday,1,1,Track Stabilization,Dynamic Track Stabilizer,Low,Clear,...,6.0,MEDIUM,0.0,1.0,1,0.555171,184.3,24.7,1,209.0
2,B08132,BMG-DTK,2023-01-01,Sunday,1,1,Ballast Renewal,Ballast Regulator,Medium,Heavy Rain,...,0.0,LOW,0.0,0.0,1,0.553304,210.2,16.7,1,226.8
3,B08111,G-NGP,2023-01-01,Sunday,1,1,Signal Maintenance,Tower Wagon,Low,Heavy Rain,...,6.0,MEDIUM,0.0,1.0,1,0.554566,116.9,44.7,1,161.6
4,B09264,KOG-RIS,2023-01-01,Sunday,1,1,Signal Maintenance,Tower Wagon,Medium,Extreme Heat,...,1.0,LOW,0.0,1.0,1,0.555311,99.4,16.5,1,116.0


## Done

`07_model_training_table.csv` is now ready. 15,000 rows, one per maintenance block, with real traffic, asset availability, section history, and calendar context all joined in.

**Next (Step 2):** load this file, drop `block_id`, `date`, `actual_duration_min` (leakage), one-hot/target-encode the categorical columns, and train a regression model on `overrun_min` with a time-based train/test split.
